## Importing the Libraries

In [47]:
import pandas as pd
import torch
from torch.utils.data import DataLoader,Dataset
import torch.nn as nn

In [48]:
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [49]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


## Tokeninzing

In [50]:
def tokenize(text):
  text = text.lower()
  text = text.replace(',', ' ')
  text = text.replace('.', ' ')
  text = text.replace('!', ' ')
  text = text.replace('?', ' ')
  text = text.replace(" ' ", ' ')
  return text.split()

In [51]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

## Vocabulary

In [52]:
vocab = {'<UNK>': 0}

In [53]:
def build_vocab(row):
  print(row['question'],row['answer'])
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:
    if token not in vocab:

      vocab[token] = len(vocab)

In [54]:
df.apply(build_vocab,axis=1)

What is the capital of France? Paris
What is the capital of Germany? Berlin
Who wrote 'To Kill a Mockingbird'? Harper-Lee
What is the largest planet in our solar system? Jupiter
What is the boiling point of water in Celsius? 100
Who painted the Mona Lisa? Leonardo-da-Vinci
What is the square root of 64? 8
What is the chemical symbol for gold? Au
Which year did World War II end? 1945
What is the longest river in the world? Nile
What is the capital of Japan? Tokyo
Who developed the theory of relativity? Albert-Einstein
What is the freezing point of water in Fahrenheit? 32
Which planet is known as the Red Planet? Mars
Who is the author of '1984'? George-Orwell
What is the currency of the United Kingdom? Pound
What is the capital of India? Delhi
Who discovered gravity? Newton
How many continents are there on Earth? 7
Which gas do plants use for photosynthesis? CO2
What is the smallest prime number? 2
Who invented the telephone? Alexander-Graham-Bell
What is the capital of Australia? Canber

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [55]:
len(vocab)

328

Now the vocabulary should be populated. Let's check its length and a few entries.

In [56]:
print(f"Vocabulary size: {len(vocab)}")
print(list(vocab.items())[:10])

Vocabulary size: 328
[('<UNK>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9)]


## Covert words into Numerical Inidces

In [57]:
def text_to_indices(text,vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text

## DataSet and DataLoader Classes

In [58]:
class Customdataset(Dataset):
  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numerical_question),torch.tensor(numerical_answer)


In [59]:
dataset = Customdataset(df,vocab)

In [60]:
dataloader = DataLoader(dataset,batch_size = 1,shuffle=True)

In [61]:
for question,answer in dataloader:
  print(question,answer)

tensor([[ 1,  2,  3, 69,  5, 53]]) tensor([[264]])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([[41]])
tensor([[ 10, 312,   3, 313, 314]]) tensor([[315]])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([[85]])
tensor([[  1,   2,   3,   4,   5, 209]]) tensor([[210]])
tensor([[ 10,  75, 211]]) tensor([[212]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 10,  75, 113]]) tensor([[114]])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[ 42, 101,   2,   3,  17]]) tensor([[102]])
tensor([[  1,   2,   3,   4,   5, 290]]) tensor([[291]])
tensor([[ 42, 259,   2, 260,  83, 261, 262]]) tensor([[263]])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([[54]])
tensor([[ 42, 316,   2, 317,  62,  63,   3, 318, 319]]) tensor([[320]])
tensor([[  1,   2,   3,   4,   5, 240, 241]]) tensor([[242]])
tensor([[  1,   2,   3, 224,   5, 225, 226, 227]]) tensor([[228]])
tensor([[ 42, 177,   2,  62,  39, 178, 179, 145, 180, 181]]) tensor([[182]])
tensor([[  1,   2,   3, 124, 125,  19,   3,  45]]) tensor

## RNN Architecture

In [62]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn = nn.RNN(input_size= 50,hidden_size=64,batch_first=True)
    self.fc = nn.Linear(in_features = 64,out_features=vocab_size)


  def forward(self,question):
    embedded_question = self.embedding(question)
    hidden,final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output

## Debugging

In [63]:
"""
  The Orginal Architecture we built:
  Problem: We are getting an Extra dimension at the end of the Architecture, Which is of No use
"""
x = nn.Embedding(328,embedding_dim = 50)
y = nn.RNN(50,64)
z = nn.Linear(64,328)

a = dataset[0][0].reshape(1,6)
print(f"Shape of A:{a.shape}")
b = x(a)
print(f"Shape of B:{b.shape}")
c,d = y(b)
print(f"Shape of C:{c.shape}")
print(f"Shape of D:{d.shape}")
e = z(d)
print(f"Shape of E:{e.shape}")

Shape of A:torch.Size([1, 6])
Shape of B:torch.Size([1, 6, 50])
Shape of C:torch.Size([1, 6, 64])
Shape of D:torch.Size([1, 6, 64])
Shape of E:torch.Size([1, 6, 328])


In [64]:
"""
This Can be Solved by placing: 'batch_size = True' in the RNN Layer,


  batch_first=True tells the RNN that the batch dimension comes first.

  batch_first=True → Input shape: (batch_size, sequence_length, input_size)
  batch_first=False (default) → Input shape: (sequence_length, batch_size, input_size)

  We finally Sequeeze the final dimentsion to extract the Output
"""
x = nn.Embedding(328,embedding_dim = 50)
y = nn.RNN(50,64,batch_first=True)
z = nn.Linear(64,328)

a = dataset[0][0].reshape(1,6)
print(f"Shape of A:{a.shape}")
b = x(a)
print(f"Shape of B:{b.shape}")
c,d = y(b)
print(f"Shape of C:{c.shape}")
print(f"Shape of D:{d.shape}")
e = z(d.squeeze(0))
print(f"Shape of E:{e.shape}")

Shape of A:torch.Size([1, 6])
Shape of B:torch.Size([1, 6, 50])
Shape of C:torch.Size([1, 6, 64])
Shape of D:torch.Size([1, 1, 64])
Shape of E:torch.Size([1, 328])


In [65]:
learning_rate = 0.001
epochs = 20

In [66]:
model = SimpleRNN(len(vocab))

In [67]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = learning_rate)

## Training Loop

In [68]:
for epoch in range(epochs):
  total_loss = 0
  for question,answer in dataloader:

    optimizer.zero_grad()
    output = model(question)

    loss = criterion(output, answer[:, 0])

    loss.backward()
    optimizer.step()

    total_loss = total_loss + loss.item()
  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 521.469569
Epoch: 2, Loss: 454.699830
Epoch: 3, Loss: 373.925199
Epoch: 4, Loss: 312.118967
Epoch: 5, Loss: 260.956579
Epoch: 6, Loss: 213.398746
Epoch: 7, Loss: 170.993869
Epoch: 8, Loss: 133.245056
Epoch: 9, Loss: 102.783178
Epoch: 10, Loss: 78.549722
Epoch: 11, Loss: 60.816005
Epoch: 12, Loss: 47.595842
Epoch: 13, Loss: 37.588580
Epoch: 14, Loss: 30.373051
Epoch: 15, Loss: 25.007213
Epoch: 16, Loss: 20.920444
Epoch: 17, Loss: 17.639562
Epoch: 18, Loss: 14.997190
Epoch: 19, Loss: 12.976244
Epoch: 20, Loss: 11.217135


## Prediction

In [69]:
def predict(model,question,threshold=0.5):

  ## Concert Question into Numbers
  numerical_question = text_to_indices(question,vocab)

  ## Adding an Extra Dimension to the Question Tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  ## Passing the Question Tensor to the Model
  output = model(question_tensor)

  ## Finding the Probability of the Each Tensor
  probs = torch.nn.functional.softmax(output, dim=-1)

  ## find the max Probability of the Answers
  value,index = torch.max(probs,dim=-1)

  ## if the valueof the max Tensor is less than threshold
  if value < threshold:
    print("I Don't Know")
    print(f"Most Probable Answer is: {list(vocab.keys())[index]}")

  else : print(list(vocab.keys())[index])

In [70]:
predict(model,"What is Capital of France?")

paris
